In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

: 

In [ ]:
# clone your branch directly
!git clone -b Valeria https://github.com/paolito81/MaskArchitectureAnomaly_CourseProject

%cd /content/MaskArchitectureAnomaly_CourseProject/eomt/

%pip install -r requirements.txt

In [ ]:
cityscapes_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"
cityscapes_data_path = "/content/drive/MyDrive/dataset_city_scapes/cityscapes"
cityscapes_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"

In [ ]:
coco_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
coco_data_path = "/content/drive/MyDrive/dataset_city_scapes/coco"
coco_ckpt_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"

In [ ]:
coco_copy_config_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_coco.yaml"

In [ ]:
default_root_dir = "/content/drive/MyDrive/eomt_runs"

In [ ]:
!git pull
!git branch -a

In [ ]:
import os

os.environ["WANDB_API_KEY"] = (
    "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"
)

In [ ]:
fine_tuned_coco_ckpt_path = "/content/drive/MyDrive/eomt_runs/epoch=0-step=1487.ckpt"
coco_copy_config_path_eval = "/content/MaskArchitectureAnomaly_CourseProject/eomt/configs/dinov2/cityscapes/semantic/eomt_base_640_finetune.yaml"

In [ ]:
!python3 main.py fit \
   -c {coco_copy_config_path_eval} \
   --trainer.max_epochs 3 \
   --model.load_ckpt_class_head False \
   --trainer.default_root_dir {default_root_dir} \
   --trainer.devices 1 \
   --data.batch_size 2 \
   --data.path {cityscapes_data_path} \
   --model.ckpt_path {coco_ckpt_path} \
   --model.load_ckpt_class_head False #False because the class head has a different number of classes in the two datasets, so we cannot load the weights from the coco checkpoint.

In [ ]:
import glob

ckpts = sorted(
    glob.glob("/content/drive/MyDrive/eomt_runs/**/*.ckpt", recursive=True)
)

print("Found checkpoints:")
for c in ckpts:
    print(c)

fine_tuned_coco_ckpt_path = ckpts[-1]

print("\nUsing latest checkpoint:")
print(fine_tuned_coco_ckpt_path)

In [ ]:
!python3 eval_shared_miou.py \
  --config {coco_copy_config_path_eval} \
  --ckpt {fine_tuned_coco_ckpt_path} \
  --cityscapes-path {cityscapes_data_path} \
  --device cuda:0 \
  --batch-size 4 \
  --num-workers 2 \
  --no-masked-attn-enabled \
  --wandb-mode online \
  --src-label-space cityscapes

In [ ]:
print("Estimated stepping batches:", self.trainer.estimated_stepping_batches)